# Embed metadata in the remaining BAGEL-family releases

Run this notebook on a free Colab CPU runtime, once per model. It does not load, cast, or quantize tensors. Set `MODEL_KEY` to `nhr`, `echo`, then `sigma`. Each run verifies the Hub LFS identity, builds variant-specific metadata, rewrites only the server working copy's header, verifies the payload SHA-256, and uploads only after the explicit gate.

`sigma` is published as an accurately described experimental artifact (`visual_und=False` plus its attribute tokens); this notebook does not claim that the current generic ComfyUI nodes implement SIGMA's multi-reference protocol.

In [ ]:
%pip install -q -U huggingface_hub hf_xet requests

import json, os, shutil, struct, subprocess
from pathlib import Path
from google.colab import userdata
from huggingface_hub import HfApi, hf_hub_download

os.environ["HF_XET_CHUNK_CACHE_SIZE_BYTES"] = "0"
REPO_ID = "6chan/bagel_comfy"
CODE_REVISION = "e053639ad9ca60314e3275462146ab8e2032a66e"
WORK_DIR = Path("/content/bagel-metadata-release")
MODEL_KEY = "nhr"  # run nhr, then echo, then sigma
UPLOAD_TO_HUB = False
HF_TOKEN = userdata.get("HF_TOKEN")
assert HF_TOKEN, "Add a write-scoped HF_TOKEN in Colab Secrets"
print({"authenticated_as": HfApi(token=HF_TOKEN).whoami()["name"], "model": MODEL_KEY, "upload_enabled": UPLOAD_TO_HUB})

In [ ]:
SIGMA_TOKENS = ["<style>", "<subject>", "<id>", "<icon>", "<text>", "<pose>", "<layout>", "<perspective>", "<lighting>", "<background>", "<environment>", "<emotion>", "<clothing>", "<texture>", "<select>"]
SPECS = {
  "nhr": {
    "filename": "bagel-nhr-edit.safetensors",
    "expected_hub_sha256": "1a29fff7b0a35b87e6dcf5197eb4ca773d619dba6c46008e5151c5ea9e10c092",
    "variant": "BAGEL-NHR-Edit", "source_repository": "iitolstykh/Bagel-NHR-Edit",
    "source_revision": "d152623c3f0be0b128f7766bc3ca0d1f3758a306",
    "source_hashes": {"ema.safetensors": "1a29fff7b0a35b87e6dcf5197eb4ca773d619dba6c46008e5151c5ea9e10c092"},
    "dtype": "bf16", "capabilities": ["image_edit"],
    "model_options": {"visual_gen": True, "visual_und": True},
    "config_repo": "iitolstykh/Bagel-NHR-Edit", "config_revision": "d152623c3f0be0b128f7766bc3ca0d1f3758a306",
    "llm_config": "llm_config.json", "vit_config": "vit_config.json"
  },
  "echo": {
    "filename": "echo-4o.safetensors",
    "expected_hub_sha256": "55f67133be7ef398ff88b78823cbae7a4e08e00a60af3c63b7a5449aecfc1445",
    "variant": "Echo-4o", "source_repository": "Yejy53/Echo-4o",
    "source_revision": "a5c35881f29f0fa6e4c6510cc1a1b88404a76034",
    "source_hashes": {"Echo-4o/ema.safetensors": "55f67133be7ef398ff88b78823cbae7a4e08e00a60af3c63b7a5449aecfc1445"},
    "dtype": "fp16", "capabilities": ["text_to_image", "multi_reference_generation"],
    "model_options": {"visual_gen": True, "visual_und": True},
    "config_repo": "Yejy53/Echo-4o", "config_revision": "a5c35881f29f0fa6e4c6510cc1a1b88404a76034",
    "llm_config": "Echo-4o/llm_config.json", "vit_config": "Echo-4o/vit_config.json"
  },
  "sigma": {
    "filename": "sigma.safetensors",
    "expected_hub_sha256": "9119f4b11cc9022a9774cc300fb5ceba293d7a2e1328bb6c2d7b7aa00fe0c813",
    "variant": "SIGMA", "source_repository": "Xiaoyan667/SIGMA-Model",
    "source_revision": "af1fd885325956d6763c45b6cfbaee9f7861f6dc",
    "source_hashes": {"ema.safetensors": "a2e3df6d5522fe2125c86a83581ca0a6113f13ad7e85c8764e82de242130479f"},
    "dtype": "bf16", "capabilities": ["multi_reference_generation"],
    "model_options": {"visual_gen": True, "visual_und": False},
    "additional_special_tokens": SIGMA_TOKENS,
    "config_repo": "ByteDance-Seed/BAGEL-7B-MoT", "config_revision": "5019f57d168e5816e8f3f701b17cc816bb7cf24b",
    "llm_config": "llm_config.json", "vit_config": "vit_config.json"
  },
}
assert MODEL_KEY in SPECS
spec = dict(SPECS[MODEL_KEY])
print(json.dumps(spec, indent=2))

In [ ]:
import requests

WORK_DIR.mkdir(parents=True, exist_ok=True)
repo_response = requests.get(f"https://huggingface.co/api/models/{REPO_ID}?blobs=true", headers={"Authorization": f"Bearer {HF_TOKEN}"}, timeout=60)
repo_response.raise_for_status()
repo_data = repo_response.json()
remote = next(item for item in repo_data["siblings"] if item["rfilename"] == spec["filename"])
assert remote["lfs"]["sha256"] == spec["expected_hub_sha256"], "Hub checkpoint changed since provenance review"
required = int(remote["size"]) + 5 * 1024**3
free = shutil.disk_usage(WORK_DIR).free
print({"free_gib": round(free/1024**3, 1), "model_gib": round(remote["size"]/1024**3, 1)})
assert free >= required, "Reconnect to a Colab runtime with enough free disk"
code_dir = WORK_DIR / "ComfyUI-BAGEL"
if not code_dir.exists():
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "https://github.com/neverbiasu/ComfyUI-BAGEL.git", str(code_dir)], check=True)
subprocess.run(["git", "-C", str(code_dir), "fetch", "--depth", "1", "origin", CODE_REVISION], check=True)
subprocess.run(["git", "-C", str(code_dir), "checkout", "--detach", CODE_REVISION], check=True)
actual_revision = subprocess.check_output(["git", "-C", str(code_dir), "rev-parse", "HEAD"], text=True).strip()
assert actual_revision == CODE_REVISION
print({"release_tool_revision": actual_revision})

In [ ]:
model_path = Path(hf_hub_download(REPO_ID, spec["filename"], local_dir=WORK_DIR, token=HF_TOKEN))
config_dir = WORK_DIR / f"configs-{MODEL_KEY}"
llm_path = Path(hf_hub_download(spec["config_repo"], spec["llm_config"], revision=spec["config_revision"], local_dir=config_dir, token=HF_TOKEN))
vit_path = Path(hf_hub_download(spec["config_repo"], spec["vit_config"], revision=spec["config_revision"], local_dir=config_dir, token=HF_TOKEN))
release_spec = {key: value for key, value in spec.items() if key not in {"filename", "expected_hub_sha256", "config_repo", "config_revision", "llm_config", "vit_config"}}
spec_path = WORK_DIR / f"{MODEL_KEY}.release-spec.json"
metadata_path = WORK_DIR / f"{MODEL_KEY}.metadata.json"
spec_path.write_text(json.dumps(release_spec, indent=2))
subprocess.run(["python", str(code_dir/"scripts/build_bagel_release_metadata.py"), "--checkpoint", str(model_path), "--spec", str(spec_path), "--llm-config", str(llm_path), "--vit-config", str(vit_path), "--output", str(metadata_path)], check=True)
subprocess.run(["python", str(code_dir/"scripts/embed_bagel_metadata.py"), "--source", str(model_path), "--metadata", str(metadata_path), "--in-place"], check=True)

In [ ]:
with model_path.open("rb") as file:
    header_len = struct.unpack("<Q", file.read(8))[0]
    header = json.loads(file.read(header_len))
embedded = json.loads(header["__metadata__"]["comfyui_bagel"])
assert embedded == json.loads(metadata_path.read_text())
assert embedded["variant"] == spec["variant"]
assert embedded["model_options"] == spec["model_options"]
assert set(embedded["model_configs"]) == {"llm_config.json", "vit_config.json"}
print({"validated": True, "variant": embedded["variant"], "tensor_count": embedded["tensor_summary"]["num_tensors"]})

In [ ]:
from huggingface_hub import CommitOperationAdd
assert UPLOAD_TO_HUB, "Local validation passed. Set UPLOAD_TO_HUB=True before publishing."
commit = HfApi(token=HF_TOKEN).create_commit(repo_id=REPO_ID, repo_type="model", operations=[CommitOperationAdd(path_in_repo=spec["filename"], path_or_fileobj=str(model_path))], commit_message=f"Embed ComfyUI-BAGEL metadata in {spec['variant']}")
print({"commit_url": commit.commit_url, "commit_oid": commit.oid})

In [ ]:
from huggingface_hub import hf_hub_url
def fetch_range(url, start, end):
    with requests.get(url, headers={"Authorization": f"Bearer {HF_TOKEN}", "Range": f"bytes={start}-{end}"}, timeout=120, stream=True) as response:
        response.raise_for_status(); assert response.status_code == 206
        data = response.raw.read(end-start+1); assert len(data) == end-start+1
        return data
remote_url = hf_hub_url(REPO_ID, spec["filename"], revision=commit.oid)
remote_len = struct.unpack("<Q", fetch_range(remote_url, 0, 7))[0]
assert remote_len <= 64*1024**2
remote_header = json.loads(fetch_range(remote_url, 8, 7+remote_len))
assert json.loads(remote_header["__metadata__"]["comfyui_bagel"]) == embedded
print({"remote_header_validated": True, "revision": commit.oid})
model_path.unlink()
shutil.rmtree(config_dir, ignore_errors=True)
print("Server working files removed; select the next MODEL_KEY.")

## Compatibility note
Keep `configs/llm_config.json` and `configs/vit_config.json` in the Hub repository for now. The new checkpoints are self-contained, but that alone does not prove every deprecated or external consumer has migrated away from the shared paths. Their removal should be a separate compatibility-reviewed release.